# 5. Model Comparison & Confusion Matrices
## Switch between all models, view confusion matrix and comparison graphs

In [ ]:
import sys
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    f1_score,
)

# Add parent for app imports (required for StackingPredictor unpickling)
sys.path.insert(0, str(Path("..").resolve()))
from app.features import RAW_FEATURES, add_derived_features, ALL_FEATURES
from app.config import AVAILABLE_MODELS, get_model_path, BEST_MODEL_JSON
import app.stacking  # noqa: F401 - needed for joblib to unpickle stacking model

In [ ]:
DATA_PATH = Path("../data/water_potability.csv")
MODELS_DIR = Path("../models")

df = pd.read_csv(DATA_PATH)
for col in RAW_FEATURES:
    if col in df.columns and df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

df = add_derived_features(df)
X = df[ALL_FEATURES]
y = df["Potability"]

_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = joblib.load(MODELS_DIR / "scaler.pkl")
X_test_s = scaler.transform(X_test)

print(f"Test set: {len(y_test)} samples")

In [ ]:
# Load all available models
MODEL_IDS = [k for k in AVAILABLE_MODELS.keys() if (MODELS_DIR / f"model_{k}.pkl").exists()]

def load_model(mid):
    return joblib.load(get_model_path(mid))

models = {mid: load_model(mid) for mid in MODEL_IDS}
print(f"Loaded {len(models)} models: {list(models.keys())}")

# Best model (default for switch)
BEST_ID = "ensemble"
if BEST_MODEL_JSON.exists():
    try:
        data = json.loads(BEST_MODEL_JSON.read_text())
        BEST_ID = data.get("best_model_id", BEST_ID)
    except Exception:
        pass
print(f"Best model (default): {AVAILABLE_MODELS.get(BEST_ID, BEST_ID)}")

## Model Switch: Confusion Matrix & ROC for Selected Model

In [ ]:
from ipywidgets import interact, Dropdown

def plot_model_metrics(model_id):
    if model_id not in models:
        print(f"Model {model_id} not found")
        return
    model = models[model_id]
    name = AVAILABLE_MODELS.get(model_id, model_id)
    
    y_pred = model.predict(X_test_s)
    proba = model.predict_proba(X_test_s)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, proba)
    f1 = f1_score(y_test, y_pred, average="weighted")
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
                xticklabels=["Not Potable", "Potable"],
                yticklabels=["Not Potable", "Potable"])
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("Actual")
    axes[0].set_title(f"Confusion Matrix - {name}")
    
    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes[1].plot(fpr, tpr, color="#3498db", lw=2, label=f"AUC = {auc:.3f}")
    axes[1].plot([0, 1], [0, 1], "k--", lw=1)
    axes[1].set_xlabel("False Positive Rate")
    axes[1].set_ylabel("True Positive Rate")
    axes[1].set_title(f"ROC Curve - {name}")
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Accuracy: {acc:.2%}  |  F1: {f1:.3f}  |  ROC-AUC: {auc:.3f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=["Not Potable", "Potable"]))

options = [(f"{AVAILABLE_MODELS.get(m, m)} {'⭐' if m == BEST_ID else ''}", m) for m in MODEL_IDS]
interact(plot_model_metrics, model_id=Dropdown(options=options, value=BEST_ID, description="Model:"))

## Comparison: Accuracy, F1, ROC-AUC Across All Models

In [ ]:
metrics = []
for mid in MODEL_IDS:
    m = models[mid]
    pred = m.predict(X_test_s)
    proba = m.predict_proba(X_test_s)[:, 1]
    metrics.append({
        "model": AVAILABLE_MODELS.get(mid, mid),
        "id": mid,
        "accuracy": accuracy_score(y_test, pred),
        "f1": f1_score(y_test, pred, average="weighted"),
        "roc_auc": roc_auc_score(y_test, proba),
    })

df_metrics = pd.DataFrame(metrics)
df_metrics = df_metrics.sort_values("accuracy", ascending=False)
df_metrics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

x = range(len(df_metrics))
colors = ["#2ecc71" if r["id"] == BEST_ID else "#3498db" for _, r in df_metrics.iterrows()]

axes[0].barh(x, df_metrics["accuracy"], color=colors)
axes[0].set_yticks(x)
axes[0].set_yticklabels(df_metrics["model"], fontsize=9)
axes[0].set_xlabel("Accuracy")
axes[0].set_title("Accuracy by Model")
axes[0].set_xlim(0, 1)

axes[1].barh(x, df_metrics["f1"], color=colors)
axes[1].set_yticks(x)
axes[1].set_yticklabels(df_metrics["model"], fontsize=9)
axes[1].set_xlabel("F1 (weighted)")
axes[1].set_title("F1 Score by Model")
axes[1].set_xlim(0, 1)

axes[2].barh(x, df_metrics["roc_auc"], color=colors)
axes[2].set_yticks(x)
axes[2].set_yticklabels(df_metrics["model"], fontsize=9)
axes[2].set_xlabel("ROC-AUC")
axes[2].set_title("ROC-AUC by Model")
axes[2].set_xlim(0, 1)

plt.tight_layout()
plt.suptitle("Model Comparison (green = best model)", y=1.02, fontsize=12)
plt.show()

## Confusion Matrices Grid (All Models)

In [ ]:
n = len(MODEL_IDS)
cols = 3
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
if rows == 1 and cols == 1:
    axes = np.array([[axes]])
elif rows == 1:
    axes = axes.reshape(1, -1)

for idx, mid in enumerate(MODEL_IDS):
    r, c = idx // cols, idx % cols
    ax = axes[r, c]
    m = models[mid]
    pred = m.predict(X_test_s)
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["N", "P"], yticklabels=["N", "P"])
    ax.set_title(f"{AVAILABLE_MODELS.get(mid, mid)}")
    ax.set_xlabel("Pred")
    ax.set_ylabel("Actual")

for idx in range(n, rows * cols):
    r, c = idx // cols, idx % cols
    axes[r, c].axis("off")

plt.tight_layout()
plt.suptitle("Confusion Matrices - All Models (N=Not Potable, P=Potable)", y=1.02)
plt.show()